# **Training Code**

In [ ]:
# Install necessary packages
!pip install ultralytics
!pip install fiftyone

import os
import torch
from ultralytics import YOLO
import fiftyone as fo
import fiftyone.zoo as foz
import yaml

# Check if CUDA is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Classes of interest
classes_of_interest = ['car', 'truck', 'motorcycle', 'bus', 'bicycle']

# Download COCO dataset (only vehicle classes)
dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    classes=classes_of_interest,
    max_samples=500  # Reduced number of samples
)

# Ensure the dataset has 'ground_truth' labels
if 'ground_truth' not in dataset.get_field_schema():
    dataset.clone_sample_field('detections', 'ground_truth')

# Shuffle and split the dataset into training and validation sets
dataset.shuffle(seed=42)
train_size = int(len(dataset) * 0.8)
# Tag the samples
dataset[:train_size].tag_samples('train')
dataset[train_size:].tag_samples('val')

# Export the dataset in YOLO format with a train/val split
export_dir = "/kaggle/working/coco_vehicles"

# Define the splits
splits = {
    'train': dataset.match_tags('train'),
    'val': dataset.match_tags('val'),
}

# Export the splits
for split_name, split_dataset in splits.items():
    split_export_dir = os.path.join(export_dir, split_name)
    split_dataset.export(
        export_dir=split_export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        classes=classes_of_interest,
        export_media=True  # Copy images to the export directory
    )

# Verify exported directories
print("Exported directories in 'export_dir':", os.listdir(export_dir))
for dirpath, dirnames, filenames in os.walk(export_dir):
    print(f"\nDirectory: {dirpath}")
    if dirnames:
        print(f" Subdirectories: {dirnames}")
    if filenames:
        print(f" Files: {filenames}")

# Adjust the paths
train_images = os.path.join(export_dir, 'train', 'images')
val_images = os.path.join(export_dir, 'val', 'images')  # Changed 'valid' to 'val'

# Check if the directories exist
if not os.path.exists(train_images):
    print(f"Training images path does not exist: {train_images}")
else:
    print(f"Training images path exists: {train_images}")

if not os.path.exists(val_images):
    print(f"Validation images path does not exist: {val_images}")
else:
    print(f"Validation images path exists: {val_images}")

# Create the data.yaml content
data_yaml = {
    'train': train_images,
    'val': val_images,  # Changed 'valid' to 'val'
    'nc': len(classes_of_interest),
    'names': classes_of_interest
}

# Save the data.yaml file
data_yaml_path = os.path.join(export_dir, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

# Load the pre-trained YOLOv5n model (nano version)
model = YOLO('yolo11s.pt')  # Using the smaller model for faster training

# Fine-tune the model
results = model.train(
    data=data_yaml_path,
    epochs=100,         # Reduced number of epochs
    imgsz=640,         # Reduced image size
    batch=32,          # Increased batch size if GPU allows
    name='yolov11s_finetune',
    pretrained=True,
    device=device,
    workers=2,         # Reduce workers if running into bottlenecks
    cache=True,        # Cache images for faster training
    verbose=False      # Suppress verbose output
)

# Save the fine-tuned model
model_path = '/kaggle/working/yolov11s_finetuned.pt'
model.save(model_path)
print(f"\nFine-tuning complete. Model saved as '{model_path}'")